In [42]:
import sympy as sp

In [43]:
from functools import lru_cache

@lru_cache(maxsize = None)
def get_ckn(k: int, n: int, p):
    if k<0 or k>n:
        return sp.Integer(0)
    if n == 0:
        return sp.Integer(1) if k==0 else sp.Integer(0)
    return get_ckn(k-1, n-1, p)/(2*p) + (k+1)*get_ckn(k+1, n-1, p)


In [44]:
x = sp.symbols('x', real = True)
alpha = sp.symbols('alpha', real = True, positive = True)
Ax = sp.symbols('Ax', real = True)

In [45]:
g0 = sp.exp(-alpha*(x-Ax)**2)
g0

exp(-alpha*(-Ax + x)**2)

In [46]:
def get_hermite_gaussian(k):
    return g0.diff(Ax, k)

In [47]:
get_hermite_gaussian(5).simplify()

-8*alpha**3*(Ax - x)*(4*alpha**2*(Ax - x)**4 - 20*alpha*(Ax - x)**2 + 15)*exp(-alpha*(Ax - x)**2)

In [48]:
nmax = 6
for n in range(nmax + 1):
    g = 0
    for k in range(0, n+1):
        g = g + get_ckn(k, n, alpha)*get_hermite_gaussian(k)
    g = g.simplify()
    print(f"g({n}) = {g}")

g(0) = exp(-alpha*(Ax - x)**2)
g(1) = (-Ax + x)*exp(-alpha*(Ax - x)**2)
g(2) = (Ax - x)**2*exp(-alpha*(Ax - x)**2)
g(3) = -Ax*(Ax - x)**2*exp(-alpha*(Ax - x)**2) + x*(Ax - x)**2*exp(-alpha*(Ax - x)**2)
g(4) = (Ax - x)**4*exp(-alpha*(Ax - x)**2)
g(5) = -Ax*(Ax - x)**4*exp(-alpha*(Ax - x)**2) + x*(Ax - x)**4*exp(-alpha*(Ax - x)**2)
g(6) = (Ax - x)**6*exp(-alpha*(Ax - x)**2)


### Implementation of overlap integrals

In [49]:
alpha, beta = sp.symbols('alpha beta', real = True, positive = True)
Ax, Ay, Az = sp.symbols('Ax Ay Az', real = True)
Bx, By, Bz = sp.symbols('Bx By Bz', real = True)

p = alpha + beta
q = alpha*beta/p
RAB2 = (Ax-Bx)**2 + (Ay-By)**2 + (Az-Bz)**2


In [50]:
S00 = (sp.sqrt(sp.pi/p))**3 * sp.exp(-q*RAB2)
S00

pi**(3/2)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(3/2)

In [51]:
T00 = sp.simplify(
    (alpha * beta / (alpha + beta))
    * (
        3
        - 2 * (alpha * beta / (alpha + beta))
        * ((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)
    )
    * (sp.pi / (alpha + beta))**sp.Rational(3, 2)
    * sp.exp(
        -(alpha * beta / (alpha + beta))
        * ((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)
    )
)
T00

pi**(3/2)*alpha*beta*(-2*alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2) + 3*alpha + 3*beta)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(7/2)

In [52]:
def build_derivative_table(base_integral, Lmax, Avars, Bvars):
    Ax, Ay, Az = Avars
    Bx, By, Bz = Bvars
    
    all_idx = [(i, j, L - i - j) for L in range(Lmax+1) 
                               for i in range(L + 1) 
                               for j in range(L+1-i)]


    @lru_cache(maxsize=None)
    def derivative(i,j,k,l,m,n):
        if (i,j,k,l,m,n) == (0,0,0,0,0,0):
            return base_integral
        if i>0:
            return sp.diff(derivative(i-1,j,k,l,m,n),Ax)
        if j>0:
            return sp.diff(derivative(i,j-1,k,l,m,n),Ay)
        if k>0:
            return sp.diff(derivative(i,j,k-1,l,m,n),Az) 
        if l>0:
            return sp.diff(derivative(i,j,k,l-1,m,n),Bx)
        if m>0:
            return sp.diff(derivative(i,j,k,l,m-1,n),By)
        return sp.diff(derivative(i,j,k,l,m,n-1),Bz) 
    
    derivatives_dict = {}
    for (i,j,k) in all_idx:
        for (l,m,n) in all_idx:
            derivatives_dict[(i,j,k,l,m,n)] = derivative(i,j,k,l,m,n)
    return derivatives_dict

In [53]:
Lmax = 1
derivatives_dict = build_derivative_table(T00, Lmax, (Ax,Ay,Az), (Bx,By,Bz))

In [54]:
for key,value in derivatives_dict.items():
    print(key, value)

(0, 0, 0, 0, 0, 0) pi**(3/2)*alpha*beta*(-2*alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2) + 3*alpha + 3*beta)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(7/2)
(0, 0, 0, 0, 0, 1) -2*pi**(3/2)*alpha**2*beta**2*(-2*Az + 2*Bz)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(7/2) - pi**(3/2)*alpha**2*beta**2*(-2*Az + 2*Bz)*(-2*alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2) + 3*alpha + 3*beta)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(9/2)
(0, 0, 0, 0, 1, 0) -2*pi**(3/2)*alpha**2*beta**2*(-2*Ay + 2*By)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(7/2) - pi**(3/2)*alpha**2*beta**2*(-2*Ay + 2*By)*(-2*alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2) + 3*alpha + 3*beta)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(9/2)
(0, 0, 0,

In [55]:
def get_integral_expressions(integral_indices, derivatives_dict, alpha, beta):
    integral_expressions = {}
    for (i,j,k,l,m,n) in integral_indices:

        ci = [get_ckn(o, i, alpha) for o in range(i+1)]
        cj = [get_ckn(o, j, alpha) for o in range(j+1)]
        ck = [get_ckn(o, k, alpha) for o in range(k+1)]
        cl = [get_ckn(o, l, beta) for o in range(l+1)]
        cm = [get_ckn(o, m, beta) for o in range(m+1)]
        cn = [get_ckn(o, n, beta) for o in range(n+1)]

        expr = 0
        for o, co in enumerate(ci):
            for p, cp in enumerate(cj):
                for q, cq in enumerate(ck):

                    for r, cr in enumerate(cl):
                        for s, cs in enumerate(cm):
                            for t, ct in enumerate(cn):
                                expr += co*cp*cq*cr*cs*ct * derivatives_dict[(o, p, q, r, s, t)]
        integral_expressions[(i,j,k,l,m,n)] = expr.simplify()
    return integral_expressions



In [56]:
def l_to_ijk(L):
    IJK = []
    for I in range(L, -1, -1):
        for J in range(L - I, -1, -1):
            IJK.append((I, J, L - I - J))
    return sorted(IJK, reverse = True)

integral_indices = []
ijk = [t for L in range(Lmax+1) for t in l_to_ijk(L)]
for i in ijk:
    for j in ijk:
        integral_indices.append(i+j)
integral_indices

[(0, 0, 0, 0, 0, 0),
 (0, 0, 0, 1, 0, 0),
 (0, 0, 0, 0, 1, 0),
 (0, 0, 0, 0, 0, 1),
 (1, 0, 0, 0, 0, 0),
 (1, 0, 0, 1, 0, 0),
 (1, 0, 0, 0, 1, 0),
 (1, 0, 0, 0, 0, 1),
 (0, 1, 0, 0, 0, 0),
 (0, 1, 0, 1, 0, 0),
 (0, 1, 0, 0, 1, 0),
 (0, 1, 0, 0, 0, 1),
 (0, 0, 1, 0, 0, 0),
 (0, 0, 1, 1, 0, 0),
 (0, 0, 1, 0, 1, 0),
 (0, 0, 1, 0, 0, 1)]

In [57]:
integral_expressions = get_integral_expressions(integral_indices, derivatives_dict, alpha, beta)

In [58]:
for key,value in integral_expressions.items():
    print(key, value)

(0, 0, 0, 0, 0, 0) pi**(3/2)*alpha*beta*(-2*alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2) + 3*alpha + 3*beta)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(7/2)
(0, 0, 0, 1, 0, 0) pi**(3/2)*alpha**2*beta*(Ax - Bx)*(-2*alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2) + 5*alpha + 5*beta)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(9/2)
(0, 0, 0, 0, 1, 0) pi**(3/2)*alpha**2*beta*(Ay - By)*(-2*alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2) + 5*alpha + 5*beta)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(9/2)
(0, 0, 0, 0, 0, 1) pi**(3/2)*alpha**2*beta*(Az - Bz)*(-2*alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2) + 5*alpha + 5*beta)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(9/2)
(1, 0, 0, 0, 0, 0) pi**(3/2)*alpha*beta**2*(2*(-Ax + Bx)*(alpha + beta) - (Ax - Bx)*(

In [59]:
integral_expressions

{(0,
  0,
  0,
  0,
  0,
  0): pi**(3/2)*alpha*beta*(-2*alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2) + 3*alpha + 3*beta)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(7/2),
 (0,
  0,
  0,
  1,
  0,
  0): pi**(3/2)*alpha**2*beta*(Ax - Bx)*(-2*alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2) + 5*alpha + 5*beta)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(9/2),
 (0,
  0,
  0,
  0,
  1,
  0): pi**(3/2)*alpha**2*beta*(Ay - By)*(-2*alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2) + 5*alpha + 5*beta)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(9/2),
 (0,
  0,
  0,
  0,
  0,
  1): pi**(3/2)*alpha**2*beta*(Az - Bz)*(-2*alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2) + 5*alpha + 5*beta)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(9/2),
 (1,
  0,
  0,
  0,
  0,
  0): pi

In [60]:
Dx, Dy, Dz, P, Q, RAB2 = sp.symbols('Dx Dy Dz P Q RAB2')
subsdict = {Ax-Bx: Dx,
            Ay-By: Dy,
            Az-Bz: Dz,
            alpha+beta: P,
            alpha*beta: Q,
            (Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2: RAB2}

In [61]:
for key, value in integral_expressions.items():
    integral_expressions[key] = sp.simplify(value.subs(subsdict, simultaneous = True))

In [62]:
for key, value in integral_expressions.items():
    print(f"Integral {key}: {value}")

Integral (0, 0, 0, 0, 0, 0): pi**(3/2)*Q*(-2*Q*RAB2 + 3*alpha + 3*beta)*exp(-Q*RAB2/P)/P**(7/2)
Integral (0, 0, 0, 1, 0, 0): pi**(3/2)*Dx*Q*alpha*(-2*Q*RAB2 + 5*alpha + 5*beta)*exp(-Q*RAB2/P)/P**(9/2)
Integral (0, 0, 0, 0, 1, 0): pi**(3/2)*Dy*Q*alpha*(-2*Q*RAB2 + 5*alpha + 5*beta)*exp(-Q*RAB2/P)/P**(9/2)
Integral (0, 0, 0, 0, 0, 1): pi**(3/2)*Dz*Q*alpha*(-2*Q*RAB2 + 5*alpha + 5*beta)*exp(-Q*RAB2/P)/P**(9/2)
Integral (1, 0, 0, 0, 0, 0): pi**(3/2)*Dx*Q*beta*(-2*P + 2*Q*RAB2 - 3*alpha - 3*beta)*exp(-Q*RAB2/P)/P**(9/2)
Integral (1, 0, 0, 1, 0, 0): -pi**(3/2)*Q*(2*Dx**2*Q*(-2*Q*RAB2 + 3*alpha + 3*beta) - 2*P**2 + P*(8*Dx**2*Q + 2*Q*RAB2 - 3*alpha - 3*beta))*exp(-Q*RAB2/P)/(2*P**(11/2))
Integral (1, 0, 0, 0, 1, 0): pi**(3/2)*Dx*Dy*Q**2*(2*Q*RAB2 - 7*alpha - 7*beta)*exp(-Q*RAB2/P)/P**(11/2)
Integral (1, 0, 0, 0, 0, 1): pi**(3/2)*Dx*Dz*Q**2*(2*Q*RAB2 - 7*alpha - 7*beta)*exp(-Q*RAB2/P)/P**(11/2)
Integral (0, 1, 0, 0, 0, 0): pi**(3/2)*Dy*Q*beta*(-2*P + 2*Q*RAB2 - 3*alpha - 3*beta)*exp(-Q*RAB2/P)

In [63]:
from sympy.printing.numpy import NumPyPrinter, _known_functions_numpy, _known_constants_numpy

In [64]:
printer = NumPyPrinter()
printer._module = "np"
printer.known_functions = {k: f"np.{v}" for k,v in _known_functions_numpy.items()}
printer.known_constants = {k: f"np.{v}" for k,v in _known_constants_numpy.items()}

In [65]:
print(printer.known_constants)

{'Exp1': 'np.e', 'Pi': 'np.pi', 'EulerGamma': 'np.euler_gamma', 'NaN': 'np.nan', 'Infinity': 'np.inf'}


In [66]:
def write_oneel_module(path, name = "S", integral_expressions = None, parameter_list = None):
    lines = ["import numpy as np",
             "from numba import njit",
             "@njit(cache = True, fastmath = True)",
             f"def {name}({parameter_list}):"]
    for key, value in integral_expressions.items():
        lines.append(f"    if (i, j, k, l, m, n) == {key}:")
        lines.append(f"        return {printer.doprint(value)}")
    
    with open(path, "w", encoding = "utf-8") as f:
        f.write("\n".join(lines))


In [69]:
from pathlib import Path
path = Path.cwd() / "T.py" 
write_oneel_module(path, name = "T", integral_expressions= integral_expressions, parameter_list = "i,j,k,l,m,n,Dx, Dy, Dz, P, Q, RAB2, alpha, beta")

In [70]:
Path.cwd()

PosixPath('/Users/rolandmitric/WORK/GITHUB/master_programming_2026/live_notebooks')